# Sound Source Localization — Experimental Results
**Paper:** *A Non-Prosthetic Assistive System for Persons with Hearing Losses: Design and Experimental Investigation*  
F. AlHayek · R. Alsubaiei · M. Alsahhaf · G. Alajmi · A. Almutairi · K. Youssef · S. Said · S. Alkork  
American University of the Middle East

---
Set **`RETRAIN = True`** to reproduce results from scratch. **`RETRAIN = False`** displays the paper's reported values.

In [ ]:
import os, copy
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'figure.dpi'       : 120,
    'savefig.dpi'      : 150,
    'savefig.bbox'     : 'tight',
})

# ── Colour palette ────────────────────────────────────────────────────────────
C_HEADER  = '#1B2A4A'   # dark navy   — table headers
C_BEST    = '#D5F0DD'   # light green — best-row highlight
C_ROW_A   = '#FFFFFF'
C_ROW_B   = '#F4F6F9'   # alternating row shading
C_BLUE    = '#2E86C1'
C_GREEN   = '#27AE60'
C_ORANGE  = '#E67E22'
C_RED     = '#C0392B'
C_GREY    = '#7F8C8D'

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT    = os.path.dirname(os.getcwd())
FEATURES_DIR = os.path.join(REPO_ROOT, '0_Dataset', 'features')
PLOTS_DIR    = os.path.join(REPO_ROOT, '4_results', 'plots')
os.makedirs(os.path.join(PLOTS_DIR, 'CNN'), exist_ok=True)
os.makedirs(os.path.join(PLOTS_DIR, 'GRU'), exist_ok=True)
os.makedirs(os.path.join(REPO_ROOT, '4_results', 'models', 'CNN'), exist_ok=True)
os.makedirs(os.path.join(REPO_ROOT, '4_results', 'models', 'GRU'), exist_ok=True)

RETRAIN   = False
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
ANGLES    = list(range(0, 360, 15))
N_CLASSES = len(ANGLES)

print(f'Device : {DEVICE} | Retrain: {RETRAIN}')

In [ ]:
# ── Shared helper: draw a publication-style table ─────────────────────────────
def draw_table(ax, headers, rows, col_widths=None, best_row=None,
               row_colors=None, header_color=C_HEADER, fontsize=10.5):
    """Render a styled table on a matplotlib Axes."""
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    n_rows = len(rows)
    n_cols = len(headers)
    if col_widths is None:
        col_widths = [1 / n_cols] * n_cols

    row_h   = 0.82 / (n_rows + 1)
    y_start = 0.95

    # cumulative x positions
    xs = [0.02]
    for w in col_widths[:-1]:
        xs.append(xs[-1] + w * 0.96)

    def cell(ax, x, y, w, h, text, fc, tc='black', fw='normal', fs=fontsize, ha='center'):
        ax.add_patch(plt.Rectangle((x, y), w * 0.96, h * 0.88,
                                   facecolor=fc, edgecolor='white', linewidth=1.2,
                                   transform=ax.transAxes, clip_on=False))
        ax.text(x + w * 0.48, y + h * 0.44, text,
                ha=ha, va='center', fontsize=fs, color=tc,
                fontweight=fw, transform=ax.transAxes)

    # header row
    for j, (h_txt, w, x) in enumerate(zip(headers, col_widths, xs)):
        cell(ax, x, y_start - row_h, w, row_h, h_txt,
             fc=header_color, tc='white', fw='bold')

    # data rows
    for i, row in enumerate(rows):
        y = y_start - row_h * (i + 2)
        if row_colors and i < len(row_colors):
            bg = row_colors[i]
        elif best_row is not None and i == best_row:
            bg = C_BEST
        else:
            bg = C_ROW_A if i % 2 == 0 else C_ROW_B
        for j, (val, w, x) in enumerate(zip(row, col_widths, xs)):
            fw = 'bold' if (best_row is not None and i == best_row) else 'normal'
            cell(ax, x, y, w, row_h, str(val), fc=bg, fw=fw)

print('Helper functions loaded.')

---
## Fig. 1 — Dataset & Feature Overview

In [ ]:
fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

# ── Left: Dataset summary table ───────────────────────────────────────────────
ax_l = fig.add_subplot(gs[0])
draw_table(
    ax_l,
    headers    = ['Parameter', 'Value'],
    col_widths = [0.50, 0.50],
    rows = [
        ['Azimuth angles', '24  (0°–345°, step 15°)'],
        ['Speaker distance', '1.75 m'],
        ['Microphone array', 'ReSpeaker XVF3800 (4 MEMS)'],
        ['Sample rate', '16 kHz'],
        ['Frame size', '16 ms  (256 samples)'],
        ['R1 — Training', '5 min / angle  (same speech)'],
        ['R2 — Validation', 'First 60 s / angle'],
        ['R2 — Test', 'Remaining 2 min / angle  (new speech)'],
        ['Silence removal', 'RMS < 50 → frame discarded'],
        ['Normalisation', 'z-score (fit on train only)'],
    ],
    fontsize = 9.5,
)
ax_l.set_title('(a) Data Collection Protocol', fontweight='bold', pad=6)

# ── Right: Feature dimensions bar chart ───────────────────────────────────────
ax_r = fig.add_subplot(gs[1])
feat_names = ['IPD\nscalar', 'IPD-Mel', 'GCC\nTDOA', 'GCC\nStrength', 'GCC\nvectors', 'Log-Mel']
feat_dims  = [3, 120, 6, 6, 600, 160]
feat_cols_c = [C_BLUE, C_BLUE, C_GREEN, C_GREEN, C_GREEN, C_ORANGE]

bars = ax_r.barh(feat_names[::-1], feat_dims[::-1], color=feat_cols_c[::-1],
                 edgecolor='white', height=0.6)
for bar, val in zip(bars, feat_dims[::-1]):
    ax_r.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
              str(val), va='center', fontsize=10, color='#333333')

ax_r.axvline(0, color='#CCCCCC', linewidth=0.8)
ax_r.set_xlabel('Dimensions')
ax_r.set_xlim(0, 720)
ax_r.set_title(f'(b) Feature Vector  —  Total: 895 dims', fontweight='bold', pad=6)

legend_patches = [
    mpatches.Patch(color=C_BLUE,   label='Phase-based (IPD)'),
    mpatches.Patch(color=C_GREEN,  label='Correlation-based (GCC)'),
    mpatches.Patch(color=C_ORANGE, label='Spectral (Log-Mel)'),
]
ax_r.legend(handles=legend_patches, fontsize=9, loc='lower right')

fig.suptitle('Fig. 1 — Dataset Protocol and Acoustic Feature Groups',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig(os.path.join(PLOTS_DIR, 'fig1_dataset_features.png'))
plt.show()

---
## Fig. 2 — Table I: CNN Feature Comparison (50 ms)

In [ ]:
# ── Paper results ─────────────────────────────────────────────────────────────
T1_FEATS    = ['GCC-Strength', 'GCC-TDOA', 'IPD', 'Log-Mel', 'All Features']
T1_ACC_30   = [85.33, 85.38, 76.35, 43.67, 88.05]
T1_F1_30    = [0.85,  0.85,  0.76,  0.44,  0.88]
T1_MAE_30   = [11.2,  10.9,  17.4,  44.0,   8.9]
T1_ACC_50   = [89.44, 90.10, 83.36, 51.21, 91.07]
T1_F1_50    = [0.90,  0.91,  0.83,  0.51,  0.91]
T1_MAE_50   = [ 8.7,   7.8,  12.5,  37.5,   6.6]

fig = plt.figure(figsize=(15, 9))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.30)

# ── (a) Full results table ─────────────────────────────────────────────────────
ax_t = fig.add_subplot(gs[0, :])
rows_t1 = [
    [f, f'{a30:.2f}%', f'{f30:.2f}', f'{m30:.1f}°', f'{a50:.2f}%', f'{f50:.2f}', f'{m50:.1f}°']
    for f, a30, f30, m30, a50, f50, m50
    in zip(T1_FEATS, T1_ACC_30, T1_F1_30, T1_MAE_30, T1_ACC_50, T1_F1_50, T1_MAE_50)
]
draw_table(
    ax_t,
    headers    = ['Feature Set', 'Acc (30ms)', 'F1 (30ms)', 'MAE (30ms)',
                               'Acc (50ms)', 'F1 (50ms)', 'MAE (50ms)'],
    col_widths = [0.22, 0.13, 0.11, 0.12, 0.13, 0.11, 0.12],
    rows       = rows_t1,
    best_row   = 4,  # 'All Features'
)
ax_t.set_title('(a) Table I — CNN Localization Performance by Feature Set',
               fontweight='bold', pad=6)

# ── (b) Accuracy comparison 30ms vs 50ms ──────────────────────────────────────
ax_b = fig.add_subplot(gs[1, 0])
x = np.arange(len(T1_FEATS))
w = 0.35
ax_b.bar(x - w/2, T1_ACC_30, width=w, color=C_BLUE,   label='30 ms', edgecolor='white')
ax_b.bar(x + w/2, T1_ACC_50, width=w, color=C_GREEN,  label='50 ms', edgecolor='white')
for i, (a30, a50) in enumerate(zip(T1_ACC_30, T1_ACC_50)):
    ax_b.text(i - w/2, a30 + 0.5, f'{a30:.0f}', ha='center', fontsize=8)
    ax_b.text(i + w/2, a50 + 0.5, f'{a50:.0f}', ha='center', fontsize=8)
ax_b.set_xticks(x)
ax_b.set_xticklabels(['GCC-Str', 'GCC-TDOA', 'IPD', 'Log-Mel', 'All'], fontsize=9)
ax_b.set_ylabel('Accuracy (%)')
ax_b.set_ylim(0, 105)
ax_b.legend(fontsize=9)
ax_b.set_title('(b) Accuracy — 30 ms vs. 50 ms', fontweight='bold')
ax_b.grid(axis='y', alpha=0.3)

# ── (c) MAE comparison ────────────────────────────────────────────────────────
ax_c = fig.add_subplot(gs[1, 1])
ax_c.bar(x - w/2, T1_MAE_30, width=w, color=C_ORANGE, label='30 ms', edgecolor='white')
ax_c.bar(x + w/2, T1_MAE_50, width=w, color=C_RED,    label='50 ms', edgecolor='white')
for i, (m30, m50) in enumerate(zip(T1_MAE_30, T1_MAE_50)):
    ax_c.text(i - w/2, m30 + 0.4, f'{m30:.0f}°', ha='center', fontsize=8)
    ax_c.text(i + w/2, m50 + 0.4, f'{m50:.0f}°', ha='center', fontsize=8)
ax_c.set_xticks(x)
ax_c.set_xticklabels(['GCC-Str', 'GCC-TDOA', 'IPD', 'Log-Mel', 'All'], fontsize=9)
ax_c.set_ylabel('Mean Absolute Error (°)')
ax_c.set_ylim(0, 55)
ax_c.legend(fontsize=9)
ax_c.set_title('(c) MAE — 30 ms vs. 50 ms  (lower is better)', fontweight='bold')
ax_c.grid(axis='y', alpha=0.3)

fig.suptitle('Fig. 2 — CNN Feature Comparison (R1 Train → R2 Test)',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig(os.path.join(PLOTS_DIR, 'fig2_cnn_feature_comparison.png'))
plt.show()

---
## Fig. 3 — Table II: Model Progression (16 ms)

In [ ]:
T2_LABELS = ['CNN\n50ms\nseq=1', 'CNN\n16ms\nseq=1', 'CNN\n16ms\nseq=2\n50% ovlp', 'GRU\n16ms\nseq=32\n50% ovlp']
T2_ACC    = [91.07, 76.34, 83.04, 99.22]
T2_MAE    = [ 6.6,  12.7,  12.4,   0.4]
T2_F1     = [0.91,  None, 0.837, 0.992]

fig = plt.figure(figsize=(15, 9))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.30)

# ── (a) Table ─────────────────────────────────────────────────────────────────
ax_t = fig.add_subplot(gs[0, :])
draw_table(
    ax_t,
    headers    = ['Model', 'Frame', 'Seq Len', 'Overlap', 'Accuracy', 'F1-Score', 'MAE'],
    col_widths = [0.14, 0.11, 0.11, 0.12, 0.14, 0.14, 0.14],
    rows = [
        ['CNN', '50 ms', '1', 'No',  '91.07%', '0.91',  '6.6°'],
        ['CNN', '16 ms', '1', 'No',  '76.34%', '—',    '12.7°'],
        ['CNN', '16 ms', '2', '50%', '83.04%', '0.837','12.4°'],
        ['GRU', '16 ms', '32','50%', '99.22%', '0.992', '0.4°'],
    ],
    best_row = 3,
)
ax_t.set_title('(a) Table II — Model Progression: CNN → GRU at 16 ms Frame Size',
               fontweight='bold', pad=6)

# ── (b) Accuracy progression ──────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[1, 0])
colors_prog = [C_BLUE, C_BLUE, C_BLUE, C_GREEN]
bars = ax_b.bar(range(4), T2_ACC, color=colors_prog, edgecolor='white', width=0.55)
bars[3].set_edgecolor(C_GREEN)
bars[3].set_linewidth(2)
for bar, val in zip(bars, T2_ACC):
    ax_b.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
              f'{val:.2f}%', ha='center', fontsize=9, fontweight='bold')
ax_b.set_xticks(range(4))
ax_b.set_xticklabels(['CNN\n50ms', 'CNN\n16ms', 'CNN-seq2\n16ms', 'GRU-seq32\n16ms'], fontsize=9)
ax_b.set_ylabel('Test Accuracy (%)')
ax_b.set_ylim(0, 112)
ax_b.set_title('(b) Accuracy Progression', fontweight='bold')
ax_b.grid(axis='y', alpha=0.3)
# Annotation arrows
for i in range(3):
    dy = T2_ACC[i+1] - T2_ACC[i]
    color = C_GREEN if dy > 0 else C_RED
    ax_b.annotate('', xy=(i+1, T2_ACC[i+1]+1), xytext=(i, T2_ACC[i]+1),
                  arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

# ── (c) MAE progression ───────────────────────────────────────────────────────
ax_c = fig.add_subplot(gs[1, 1])
bars_m = ax_c.bar(range(4), T2_MAE, color=colors_prog, edgecolor='white', width=0.55)
bars_m[3].set_edgecolor(C_GREEN); bars_m[3].set_linewidth(2)
for bar, val in zip(bars_m, T2_MAE):
    ax_c.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
              f'{val:.1f}°', ha='center', fontsize=9, fontweight='bold')
ax_c.set_xticks(range(4))
ax_c.set_xticklabels(['CNN\n50ms', 'CNN\n16ms', 'CNN-seq2\n16ms', 'GRU-seq32\n16ms'], fontsize=9)
ax_c.set_ylabel('MAE (°)  — lower is better')
ax_c.set_ylim(0, 17)
ax_c.set_title('(c) MAE Progression', fontweight='bold')
ax_c.grid(axis='y', alpha=0.3)

legend_p = [
    mpatches.Patch(color=C_BLUE,  label='CNN baseline'),
    mpatches.Patch(color=C_GREEN, label='GRU — best model'),
]
fig.legend(handles=legend_p, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.04))

fig.suptitle('Fig. 3 — Model Progression: Adapting to 16 ms Native Frame Size',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig(os.path.join(PLOTS_DIR, 'fig3_model_progression.png'))
plt.show()

---
## Fig. 4 — Table III: Real-Time Evaluation

In [ ]:
fig = plt.figure(figsize=(15, 8))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.32)

# ── (a) Table ─────────────────────────────────────────────────────────────────
ax_t = fig.add_subplot(gs[0, :])
draw_table(
    ax_t,
    headers    = ['Model', 'Frames Evaluated', 'Exact Accuracy', '±15° Accuracy', 'MAE'],
    col_widths = [0.28, 0.18, 0.18, 0.18, 0.14],
    rows = [
        ['CNN  (16 ms, seq=2)',  '5,764', '35.4%', '82.9%', '16.8°'],
        ['GRU  (16 ms, seq=32)', '5,320', '46.1%', '90.7%',  '9.5°'],
    ],
    best_row = 1,
)
ax_t.set_title('(a) Table III — Real-Time Localization Performance on Live Audio',
               fontweight='bold', pad=6)

# ── (b) Accuracy: exact vs ±15° ───────────────────────────────────────────────
ax_b = fig.add_subplot(gs[1, 0])
models_rt = ['CNN', 'GRU']
exact_rt  = [35.4, 46.1]
pm15_rt   = [82.9, 90.7]
x_rt = np.arange(2)
w = 0.32
ax_b.bar(x_rt - w/2, exact_rt, width=w, color=[C_BLUE, C_GREEN],   label='Exact acc', edgecolor='white')
ax_b.bar(x_rt + w/2, pm15_rt,  width=w, color=[C_BLUE, C_GREEN],   label='±15° acc',
         edgecolor='white', alpha=0.55, hatch='//')
for i, (e, p) in enumerate(zip(exact_rt, pm15_rt)):
    ax_b.text(i - w/2, e + 0.5, f'{e}%', ha='center', fontsize=9, fontweight='bold')
    ax_b.text(i + w/2, p + 0.5, f'{p}%', ha='center', fontsize=9, fontweight='bold')
ax_b.set_xticks(x_rt); ax_b.set_xticklabels(models_rt, fontsize=11)
ax_b.set_ylabel('Accuracy (%)')
ax_b.set_ylim(0, 110)
ax_b.legend(fontsize=9)
ax_b.set_title('(b) Exact vs. ±15° Accuracy', fontweight='bold')
ax_b.grid(axis='y', alpha=0.3)

# ── (c) MAE comparison ────────────────────────────────────────────────────────
ax_c = fig.add_subplot(gs[1, 1])
mae_rt = [16.8, 9.5]
bars_c = ax_c.bar(x_rt, mae_rt, color=[C_ORANGE, C_GREEN], edgecolor='white', width=0.45)
for bar, val in zip(bars_c, mae_rt):
    ax_c.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
              f'{val}°', ha='center', fontsize=11, fontweight='bold')
ax_c.set_xticks(x_rt); ax_c.set_xticklabels(models_rt, fontsize=11)
ax_c.set_ylabel('MAE (°)')
ax_c.set_ylim(0, 22)
ax_c.set_title('(c) MAE — Real-Time', fontweight='bold')
ax_c.grid(axis='y', alpha=0.3)

# ── (d) Offline vs Real-Time full comparison ──────────────────────────────────
ax_d = fig.add_subplot(gs[1, 2])
cond_labels = ['CNN\nOffline', 'CNN\nReal-Time', 'GRU\nOffline', 'GRU\nReal-Time']
cond_acc    = [76.34, 35.4, 99.22, 46.1]
cond_colors = [C_BLUE, C_BLUE, C_GREEN, C_GREEN]
cond_alpha  = [1.0, 0.55, 1.0, 0.55]
bars_d = ax_d.bar(range(4), cond_acc, color=cond_colors, edgecolor='white',
                  width=0.55)
for bar, alpha in zip(bars_d, cond_alpha):
    bar.set_alpha(alpha)
for bar, val in zip(bars_d, cond_acc):
    ax_d.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
              f'{val}%', ha='center', fontsize=8.5, fontweight='bold')
ax_d.set_xticks(range(4))
ax_d.set_xticklabels(cond_labels, fontsize=8.5)
ax_d.set_ylabel('Exact Accuracy (%)')
ax_d.set_ylim(0, 118)
ax_d.set_title('(d) Offline vs. Real-Time', fontweight='bold')
ax_d.grid(axis='y', alpha=0.3)

fig.suptitle('Fig. 4 — Real-Time Localization Evaluation on Live ReSpeaker Audio',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig(os.path.join(PLOTS_DIR, 'fig4_realtime_evaluation.png'))
plt.show()

---
## Fig. 5 — GRU Training (RETRAIN=True only)

In [ ]:
# ── Model & data definitions (needed when RETRAIN=True) ───────────────────────
MICS         = ['mic_right', 'mic_front', 'mic_left', 'mic_back']
N_MELS       = 40; GCC_VEC_SIZE = 100; N_PAIRS = 6
RMS_COLS     = [f'rms_{m}' for m in MICS]
LOGMEL_COLS  = [f'logmel_{m}_b{b}' for m in MICS for b in range(N_MELS)]
ALL_FEAT     = [
    'ipd_pair0','ipd_pair1','ipd_pair2',
    *[f'ipd_mel_{i}_b{b}'  for i in range(3)      for b in range(N_MELS)],
    *[f'gcc_tdoa_{i}'      for i in range(N_PAIRS)],
    *[f'gcc_strength_{i}'  for i in range(N_PAIRS)],
    *[f'gcc_vec_{i}_t{t}'  for i in range(N_PAIRS) for t in range(GCC_VEC_SIZE)],
    *LOGMEL_COLS,
]

def angular_metrics(y_true, y_pred):
    td  = np.array([ANGLES[i] for i in y_true], dtype=np.float32)
    pd_ = np.array([ANGLES[i] for i in y_pred], dtype=np.float32)
    diff = np.minimum(np.abs(td - pd_), 360 - np.abs(td - pd_))
    return float(np.mean(diff)), float(np.sqrt(np.mean(diff**2)))

def eval_loader(model, loader):
    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            preds.append(model(xb.to(DEVICE)).argmax(1).cpu()); labels.append(yb)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()

class SequenceGRU(nn.Module):
    def __init__(self, n_features, embed_dim=256, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.embed = nn.Sequential(nn.Linear(n_features, embed_dim), nn.LayerNorm(embed_dim), nn.ReLU(), nn.Dropout(dropout))
        self.gru   = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.clf   = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Dropout(dropout), nn.Linear(hidden_dim, N_CLASSES))
    def forward(self, x): return self.clf(self.gru(self.embed(x))[0][:, -1, :])

class SequenceDataset(Dataset):
    def __init__(self, X, y, chunks, positions, seq_len, stride=1):
        self.X = X.astype(np.float32); self.y = y; self.seq = seq_len
        self.starts = self._make_starts(y, chunks, positions, stride)
    def _make_starts(self, y, chunks, positions, stride):
        starts = []
        for pos in np.unique(positions):
            idx = np.flatnonzero(positions == pos)
            for off in range(0, len(idx)-self.seq+1, stride):
                win = idx[off:off+self.seq]
                if not (np.all(np.diff(win)==1) and np.all(y[win]==y[win[0]]) and np.all(np.diff(chunks[win])==1)): continue
                starts.append(int(win[0]))
        return np.array(starts, dtype=np.int64)
    def __len__(self): return len(self.starts)
    def __getitem__(self, i):
        s = self.starts[i]
        return torch.from_numpy(self.X[s:s+self.seq]), torch.tensor(int(self.y[s]), dtype=torch.long)

print('Model classes defined.')

In [ ]:
if not RETRAIN:
    print('RETRAIN=False — skipping training. Set RETRAIN=True to reproduce GRU results.')
else:
    # Load data
    train_df = pd.read_csv(os.path.join(FEATURES_DIR, 'train_DATA16.csv'))
    test_df  = pd.read_csv(os.path.join(FEATURES_DIR, 'test_DATA16.csv'))
    for df in [train_df, test_df]:
        mask = (df[RMS_COLS] >= 50.0).all(axis=1)
        df.drop(index=df.index[~mask], inplace=True); df.reset_index(drop=True, inplace=True)

    val_chunks = 60 * 1000 // 16
    val_p, te_p = [], []
    for lbl in sorted(test_df['label'].unique()):
        adf = test_df[test_df['label']==lbl].sort_values('chunk')
        val_p.append(adf.iloc[:val_chunks]); te_p.append(adf.iloc[val_chunks:])
    val_df = pd.concat(val_p, ignore_index=True)
    test_df = pd.concat(te_p, ignore_index=True)

    def arrays(df):
        X = df[ALL_FEAT].values.astype(np.float32)
        y = df['label'].to_numpy(dtype=np.int64)
        c = df['chunk'].to_numpy(dtype=np.int64)
        p = df['position'].to_numpy(dtype=np.int64) if 'position' in df.columns else np.zeros(len(df), np.int64)
        return X, y, c, p

    X_tr, y_tr, c_tr, p_tr = arrays(train_df)
    X_va, y_va, c_va, p_va = arrays(val_df)
    X_te, y_te, c_te, p_te = arrays(test_df)

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr).astype(np.float32)
    X_va = scaler.transform(X_va).astype(np.float32)
    X_te = scaler.transform(X_te).astype(np.float32)

    tr_ds = SequenceDataset(X_tr, y_tr, c_tr, p_tr, 32, 16)
    va_ds = SequenceDataset(X_va, y_va, c_va, p_va, 32, 32)
    te_ds = SequenceDataset(X_te, y_te, c_te, p_te, 32, 32)
    tr_ld = DataLoader(tr_ds, 256, shuffle=True,  pin_memory=True)
    va_ld = DataLoader(va_ds, 512, shuffle=False, pin_memory=True)
    te_ld = DataLoader(te_ds, 512, shuffle=False, pin_memory=True)

    model     = SequenceGRU(X_tr.shape[1]).to(DEVICE)
    opt       = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-4)
    sched     = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.03)

    best_val, best_state, patience_cnt, history = -1, None, 0, []
    for epoch in range(50):
        model.train(); correct, total, loss_sum = 0, 0, 0.0
        for xb, yb in tr_ld:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); out = model(xb); loss = criterion(out, yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            loss_sum += loss.item()*len(yb); correct += (out.argmax(1)==yb).sum().item(); total += len(yb)
        sched.step()
        vp, vl = eval_loader(model, va_ld); val_acc = (vp==vl).mean()
        history.append({'epoch':epoch+1,'loss':loss_sum/total,'train_acc':correct/total,'val_acc':float(val_acc)})
        if val_acc > best_val: best_val=val_acc; best_state=copy.deepcopy(model.state_dict()); patience_cnt=0
        else: patience_cnt+=1
        if patience_cnt>=8: print(f'Early stop epoch {epoch+1}'); break
        if (epoch+1)%5==0: print(f'Epoch {epoch+1:>3}  loss={loss_sum/total:.4f}  train={correct/total*100:.1f}%  val={val_acc*100:.1f}%')

    model.load_state_dict(best_state)
    gru_preds, gru_true = eval_loader(model, te_ld)
    gru_acc = (gru_preds==gru_true).mean()*100
    gru_mae, _ = angular_metrics(gru_true, gru_preds)
    print(f'\nGRU Test: Acc={gru_acc:.2f}%  MAE={gru_mae:.1f}°')

    # ── Fig 5: Training curves ──────────────────────────────────────────────────
    fig = plt.figure(figsize=(14, 5))
    gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.30)

    ep_x = [h['epoch'] for h in history]
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(ep_x, [h['train_acc'] for h in history], color=C_BLUE,  lw=2, label='Train')
    ax1.plot(ep_x, [h['val_acc']   for h in history], color=C_GREEN, lw=2, label='Validation')
    ax1.axhline(best_val, color=C_GREEN, linestyle=':', lw=1.2, alpha=0.6)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy'); ax1.set_ylim(0, 1.05)
    ax1.set_title('(a) Accuracy Curves', fontweight='bold'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2 = fig.add_subplot(gs[1])
    ax2.plot(ep_x, [h['loss'] for h in history], color=C_ORANGE, lw=2)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Cross-Entropy Loss')
    ax2.set_title('(b) Training Loss', fontweight='bold'); ax2.grid(True, alpha=0.3)

    fig.suptitle(f'Fig. 5 — GRU Training Curves  |  Test Acc = {gru_acc:.2f}%   MAE = {gru_mae:.1f}°',
                 fontsize=13, fontweight='bold')
    plt.savefig(os.path.join(PLOTS_DIR, 'GRU', 'fig5_gru_training.png'))
    plt.show()

---
## Fig. 6 — GRU Confusion Matrix & Per-Angle Accuracy (RETRAIN=True only)

In [ ]:
if not RETRAIN:
    print('RETRAIN=False — run with RETRAIN=True to generate confusion matrix.')
else:
    cm = confusion_matrix(gru_true, gru_preds)
    class_acc = [(gru_preds[gru_true==i]==i).mean()*100 if (gru_true==i).any() else 0
                 for i in range(N_CLASSES)]

    fig = plt.figure(figsize=(15, 6))
    gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.30)

    ax1 = fig.add_subplot(gs[0])
    im  = ax1.imshow(cm, cmap='Blues', aspect='auto')
    ax1.set_xticks(range(N_CLASSES)); ax1.set_yticks(range(N_CLASSES))
    ax1.set_xticklabels(ANGLES, rotation=90, fontsize=7)
    ax1.set_yticklabels(ANGLES, fontsize=7)
    ax1.set_xlabel('Predicted Angle (°)'); ax1.set_ylabel('True Angle (°)')
    ax1.set_title(f'(a) Confusion Matrix  —  Acc = {gru_acc:.2f}%', fontweight='bold')
    plt.colorbar(im, ax=ax1, fraction=0.04)

    ax2 = fig.add_subplot(gs[1])
    clrs = [C_GREEN if v >= 95 else C_BLUE if v >= 80 else C_ORANGE for v in class_acc]
    ax2.bar(ANGLES, class_acc, width=12, color=clrs, edgecolor='white')
    ax2.axhline(gru_acc, color=C_RED, linestyle='--', lw=1.5, label=f'Overall = {gru_acc:.1f}%')
    ax2.set_xlabel('Angle (°)'); ax2.set_ylabel('Accuracy (%)')
    ax2.set_ylim(0, 112); ax2.set_xticks(ANGLES)
    ax2.set_xticklabels(ANGLES, rotation=90, fontsize=7)
    ax2.set_title(f'(b) Per-Angle Accuracy  —  MAE = {gru_mae:.1f}°', fontweight='bold')
    ax2.legend(); ax2.grid(axis='y', alpha=0.3)

    fig.suptitle('Fig. 6 — GRU Model: Confusion Matrix and Per-Angle Accuracy',
                 fontsize=13, fontweight='bold')
    plt.savefig(os.path.join(PLOTS_DIR, 'GRU', 'fig6_gru_confusion.png'))
    plt.show()

---
## Fig. 7 — Complete Results Summary

In [ ]:
fig = plt.figure(figsize=(15, 7))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

# ── (a) Summary table ─────────────────────────────────────────────────────────
ax_t = fig.add_subplot(gs[0])
draw_table(
    ax_t,
    headers    = ['Stage', 'Acc', 'MAE', 'Note'],
    col_widths = [0.44, 0.16, 0.14, 0.22],
    rows = [
        ['CNN 50ms — GCC-Str',      '89.44%',  '8.7°',  'Feature study'],
        ['CNN 50ms — GCC-TDOA',     '90.10%',  '7.8°',  'Feature study'],
        ['CNN 50ms — IPD',          '83.36%', '12.5°',  'Feature study'],
        ['CNN 50ms — Log-Mel',      '51.21%', '37.5°',  'Feature study'],
        ['CNN 50ms — All Features', '91.07%',  '6.6°',  'Baseline'],
        ['CNN 16ms — seq=1',        '76.34%', '12.7°',  'Frame reduced'],
        ['CNN 16ms — seq=2, 50%',   '83.04%', '12.4°',  'Overlap added'],
        ['GRU 16ms — seq=32, 50%',  '99.22%',  '0.4°',  '★ Best model'],
    ],
    best_row = 7,
    fontsize  = 9.5,
)
ax_t.set_title('(a) All Localization Results — Paper Reference', fontweight='bold', pad=6)

# ── (b) Accuracy & MAE dual-axis ──────────────────────────────────────────────
ax_b = fig.add_subplot(gs[1])
stages   = ['CNN\n50ms\nAll', 'CNN\n16ms\nseq=1', 'CNN\n16ms\nseq=2', 'GRU\n16ms\nseq=32']
acc_vals = [91.07, 76.34, 83.04, 99.22]
mae_vals = [ 6.6,  12.7,  12.4,   0.4]
clr_bars = [C_BLUE, C_BLUE, C_BLUE, C_GREEN]

ax_b2 = ax_b.twinx()

x = np.arange(4)
bars_acc = ax_b.bar(x - 0.18, acc_vals, width=0.32, color=clr_bars, alpha=0.9,
                    edgecolor='white', label='Accuracy (%)')
bars_mae = ax_b2.bar(x + 0.18, mae_vals, width=0.32, color=clr_bars, alpha=0.45,
                     edgecolor='white', hatch='xx', label='MAE (°)')

for bar, val in zip(bars_acc, acc_vals):
    ax_b.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
              f'{val:.0f}%', ha='center', fontsize=8)
for bar, val in zip(bars_mae, mae_vals):
    ax_b2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
               f'{val:.1f}°', ha='center', fontsize=8)

ax_b.set_xticks(x); ax_b.set_xticklabels(stages, fontsize=9)
ax_b.set_ylabel('Accuracy (%)', color=C_BLUE)
ax_b2.set_ylabel('MAE (°)  — lower is better', color=C_ORANGE)
ax_b.set_ylim(0, 120); ax_b2.set_ylim(0, 18)
ax_b.set_title('(b) Accuracy vs. MAE — Model Progression', fontweight='bold')
ax_b.grid(axis='y', alpha=0.3)

lines1, labels1 = ax_b.get_legend_handles_labels()
lines2, labels2 = ax_b2.get_legend_handles_labels()
ax_b.legend(lines1+lines2, labels1+labels2, fontsize=9, loc='upper left')

fig.suptitle('Fig. 7 — Localization Results Summary',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig(os.path.join(PLOTS_DIR, 'fig7_localization_summary.png'))
plt.show()
print('All figures saved to:', PLOTS_DIR)